# Qwen3 0.6B Twitter Sentiment Classification — Sequence Classification Head Only

This notebook is a focused version of the previous notebook.

It does **not** use generative classification and does **not** use `SFTTrainer`.

Instead, it uses:

- `AutoModelForSequenceClassification`
- a classification head with 2 labels: `negative`, `positive`
- optional LoRA using `TaskType.SEQ_CLS`
- Hugging Face `Trainer`
- direct classifier logits and probabilities

This is the right path if your goal is a real classifier instead of asking a causal LM to generate the label text.

## 0. Environment check

Your current environment is:

- Python 3.11.10
- Torch 2.4.1 + CUDA 12.4
- GPU: NVIDIA RTX 5000 Ada Generation
- Compute capability: 8.9

This is fine for RTX 5000 Ada. The CUDA test should pass before training.

In [1]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Torch arch list:", torch.cuda.get_arch_list())

    x = torch.randn(4, 4, device="cuda")
    y = x @ x
    print("CUDA test passed:")
    print(y)
else:
    print("CUDA is not available. Training will run on CPU and will be slow.")

Python: 3.11.10 (main, Sep  7 2024, 18:35:41) [GCC 11.4.0]
Torch: 2.4.1+cu124
Torch CUDA: 12.4
CUDA available: True
GPU: NVIDIA RTX 5000 Ada Generation
Compute capability: (8, 9)
Torch arch list: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
CUDA test passed:
tensor([[-0.5905, -2.8019, -4.6182, -0.5483],
        [-1.1947,  5.1274,  0.6937, -0.8054],
        [ 2.6419,  0.4566,  0.2840, -1.1684],
        [ 0.9933, -0.6895, -1.7929,  1.6684]], device='cuda:0')


## 1. Install required libraries

Run this cell only if packages are missing or incompatible.

For your current Torch setup, this notebook intentionally does **not** reinstall Torch.

Recommended:
- Keep `torch==2.4.1+cu124`
- Use a Qwen3-compatible Transformers version
- Avoid installing random latest nightly builds unless needed

If you previously faced `torch.float8_e8m0fnu` errors with very new Transformers versions, pinning `transformers==4.51.3` is safer with this Torch version.

In [2]:
# Optional installation cell.
# Run only if needed.

%pip install -U --no-cache-dir \
  "transformers==4.51.3" \
  "accelerate>=1.6.0" \
  "peft>=0.15.0" \
  "datasets" \
  "evaluate" \
  "scikit-learn" \
  "pandas" \
  "tqdm"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 297.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.6.0 requires transformers>=4.56.2, but y

## 2. Imports and global configuration

We define:

- label mapping
- base model name
- output directories
- training precision
- seed
- device

In [3]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import load_dataset, DatasetDict
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ID2LABEL = {
    0: "negative",
    1: "positive",
}

LABEL2ID = {
    "negative": 0,
    "positive": 1,
}

MODEL_NAME = "Qwen/Qwen3-0.6B"

CLS_LORA_OUTPUT_DIR = "./qwen3-0.6b-twitter-sentiment-classifier-lora"
CLS_LORA_SAVED_DIR = "./qwen3-0.6b-twitter-sentiment-classifier-lora_saved"

DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

print("Device:", DEVICE)
print("Using dtype:", DTYPE)

Device: cuda
Using dtype: torch.bfloat16


## 3. Load the Twitter sentiment dataset

This uses the same source as your previous notebook.

The raw files have tab-separated rows:

```text
label<TAB>tweet_text
```

The final dataset has:

- `text`
- `feeling`

where:

- `0 = negative`
- `1 = positive`

In [4]:
base_url = "https://raw.githubusercontent.com/cblancac/SentimentAnalysisBert/main/data/"

data_files = {
    "train_raw": base_url + "train_150k.txt",
    "test": base_url + "test_62k.txt",
}

raw_ds = load_dataset("text", data_files=data_files)

def parse_tweet(example):
    label, text = example["text"].split("\t", 1)
    return {
        "text": text.strip(),
        "feeling": int(label),
    }

parsed_ds = raw_ds.map(
    parse_tweet,
    remove_columns=["text"],
)

train_valid = parsed_ds["train_raw"].train_test_split(
    train_size=0.8,
    seed=SEED,
)

ds = DatasetDict({
    "train": train_valid["train"],
    "validation": train_valid["test"],
    "test": parsed_ds["test"],
})

print(ds)
print(ds["train"][0])
print(ds["train"].features)

Generating train_raw split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/149985 [00:00<?, ? examples/s]

Map:   0%|          | 0/61998 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'feeling'],
        num_rows: 119988
    })
    validation: Dataset({
        features: ['text', 'feeling'],
        num_rows: 29997
    })
    test: Dataset({
        features: ['text', 'feeling'],
        num_rows: 61998
    })
})
{'text': '@fa6ami86 so happy that salman won.  btw the 14sec clip is truely a teaser', 'feeling': 0}
{'text': Value('string'), 'feeling': Value('int64')}


## 4. Optional: create smaller training/evaluation subsets

Use this when experimenting.

For final training, set:

```python
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None
```

For fast debugging, keep the values smaller.

In [5]:
MAX_TRAIN_SAMPLES = 20_000
MAX_VALID_SAMPLES = 5_000
MAX_TEST_SAMPLES = 5_000

def maybe_select_subset(dataset, max_samples, seed=SEED):
    if max_samples is None:
        return dataset
    max_samples = min(max_samples, len(dataset))
    return dataset.shuffle(seed=seed).select(range(max_samples))

work_ds = DatasetDict({
    "train": maybe_select_subset(ds["train"], MAX_TRAIN_SAMPLES),
    "validation": maybe_select_subset(ds["validation"], MAX_VALID_SAMPLES),
    "test": maybe_select_subset(ds["test"], MAX_TEST_SAMPLES),
})

print(work_ds)
print(work_ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'feeling'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'feeling'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'feeling'],
        num_rows: 5000
    })
})
{'text': 'Dhani Harrison got rid of his twitter. Woe is me', 'feeling': 0}


## 5. Load Qwen3 tokenizer

Qwen tokenizer may not have a dedicated pad token.

For sequence classification with batched inputs, we set:

```python
pad_token = eos_token
```

and later also set:

```python
model.config.pad_token_id = tokenizer.pad_token_id
```

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

pad_token: <|endoftext|>
pad_token_id: 151643
eos_token: <|im_end|>
eos_token_id: 151645


## 6. Load Qwen3 with a sequence-classification head

This is the core change.

Instead of:

```python
AutoModelForCausalLM
```

we use:

```python
AutoModelForSequenceClassification
```

The model now outputs:

```python
logits = [negative_logit, positive_logit]
```

The classification head will be newly initialized, so a warning about `score.weight` being newly initialized is normal.

In [7]:
def load_sequence_classification_base_model():
    # Loads Qwen3 with a sequence-classification head.
    # Uses torch_dtype for compatibility with Transformers 4.51.x.
    # If your Transformers version only accepts dtype, replace torch_dtype with dtype.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        torch_dtype=DTYPE,
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.problem_type = "single_label_classification"
    model.config.use_cache = False

    return model

base_cls_model = load_sequence_classification_base_model()

print("Loaded base sequence-classification model.")
print("Model class:", type(base_cls_model))

print("\nClassifier-related modules:")
for name, module in base_cls_model.named_modules():
    if "score" in name or "classifier" in name:
        print(name, module)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded base sequence-classification model.
Model class: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForSequenceClassification'>

Classifier-related modules:
score Linear(in_features=1024, out_features=2, bias=False)


## 7. Add LoRA for sequence classification

For Qwen-style transformer blocks, we attach LoRA to attention and MLP projection layers:

- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`
- `gate_proj`
- `up_proj`
- `down_proj`

We also save and train the classification head using:

```python
modules_to_save=["score"]
```

This is important because the classifier head is task-specific and newly initialized.

In [8]:
clf_peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    modules_to_save=["score"],
)

clf_model = get_peft_model(base_cls_model, clf_peft_config)
clf_model.print_trainable_parameters()

clf_model.to(DEVICE)

trainable params: 10,094,592 || all params: 606,146,560 || trainable%: 1.6654


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Qwen3ForSequenceClassification(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

## 8. Tokenize the dataset for classification

For classifier-head training, the target label should be in a column called:

```python
labels
```

So we rename:

```python
feeling -> labels
```

We keep only:

- `input_ids`
- `attention_mask`
- `labels`

In [9]:
MAX_LENGTH = 256

def tokenize_for_classification(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

clf_ds = work_ds.map(
    tokenize_for_classification,
    batched=True,
)

clf_ds = clf_ds.rename_column("feeling", "labels")

columns_to_keep = {"input_ids", "attention_mask", "labels"}
columns_to_remove = [
    col for col in clf_ds["train"].column_names
    if col not in columns_to_keep
]

clf_ds = clf_ds.remove_columns(columns_to_remove)

print(clf_ds)
print(clf_ds["train"][0])

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
})
{'labels': 0, 'input_ids': [35, 86811, 35527, 2684, 9279, 315, 806, 22272, 13, 467, 4644, 374, 752], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## 9. Data collator and metrics

`DataCollatorWithPadding` dynamically pads each batch to the longest sequence in that batch.

This is better than padding every tweet to `MAX_LENGTH`.

In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

## 10. Training configuration

For your RTX 5000 Ada Generation GPU:

- BF16 should usually work
- TF32 is enabled
- `adamw_torch_fused` is used on GPU
- Batch size 32 is a reasonable starting point for Qwen3-0.6B

If you get OOM, reduce:

```python
per_device_train_batch_size
per_device_eval_batch_size
```

If GPU utilization is low, increase batch size to 48 or 64.

In [13]:
clf_training_args = TrainingArguments(
    output_dir=CLS_LORA_OUTPUT_DIR,

    num_train_epochs=20,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,

    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,

    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    tf32=True,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    logging_steps=25,
    report_to="none",

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    seed=SEED,
)

## 11. Train the sequence-classification-head LoRA model

This trains:

- LoRA adapter weights
- the classification head `score`

The base Qwen weights remain frozen.

In [14]:
clf_trainer = Trainer(
    model=clf_model,
    args=clf_training_args,
    train_dataset=clf_ds["train"],
    eval_dataset=clf_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_cls_metrics,
)

train_result = clf_trainer.train()
print(train_result)

/tmp/ipykernel_1470/207719773.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  clf_trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokeni

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
500,0.384900,0.441524,0.825000,0.824133,0.824109
1000,0.229200,0.477684,0.823000,0.822415,0.822395
1500,0.114200,0.700809,0.838000,0.837680,0.837694
2000,0.091600,0.903861,0.831800,0.831744,0.831750
2500,0.079500,0.774652,0.831800,0.831465,0.831480
3000,0.087900,0.897889,0.827400,0.827391,0.827388
3500,0.078500,0.875451,0.829400,0.829385,0.829388
4000,0.043600,1.202634,0.835000,0.834683,0.834697
4500,0.008800,1.336653,0.830800,0.830221,0.830241
5000,0.030800,1.209158,0.840800,0.840792,0.840789


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

KeyboardInterrupt: 

## 12. Evaluate on validation and test sets

This gives clean classifier metrics from logits, not generated text.

In [15]:
valid_metrics = clf_trainer.evaluate(clf_ds["validation"])
print("Validation metrics:")
print(valid_metrics)

test_metrics = clf_trainer.evaluate(clf_ds["test"])
print("\nTest metrics:")
print(test_metrics)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Validation metrics:
{'eval_loss': 1.552922248840332, 'eval_accuracy': 0.8388, 'eval_f1_macro': 0.8387811954386359, 'eval_f1_weighted': 0.8387777131124574}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


Test metrics:
{'eval_loss': 1.5660656690597534, 'eval_accuracy': 0.8398, 'eval_f1_macro': 0.8396760952734712, 'eval_f1_weighted': 0.8397313621299085}


## 13. Detailed test-set report

This section produces:

- accuracy
- classification report
- confusion matrix
- dataframe with predictions and probabilities

In [16]:
pred_output = clf_trainer.predict(clf_ds["test"])

test_logits = pred_output.predictions
test_labels = pred_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

print("Accuracy:", accuracy_score(test_labels, test_preds))

print("\nClassification report:")
print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["negative", "positive"],
        zero_division=0,
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(test_labels, test_preds))

test_probs = torch.softmax(torch.tensor(test_logits).float(), dim=-1).numpy()

test_records = []
raw_test_rows = work_ds["test"]

for i in range(len(test_preds)):
    test_records.append({
        "text": raw_test_rows[i]["text"],
        "true_label": ID2LABEL[int(test_labels[i])],
        "pred_label": ID2LABEL[int(test_preds[i])],
        "negative_prob": float(test_probs[i][0]),
        "positive_prob": float(test_probs[i][1]),
        "correct": bool(test_labels[i] == test_preds[i]),
    })

results_df = pd.DataFrame(test_records)
results_df.head()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Accuracy: 0.8398

Classification report:
              precision    recall  f1-score   support

    negative       0.85      0.82      0.84      2469
    positive       0.83      0.86      0.84      2531

    accuracy                           0.84      5000
   macro avg       0.84      0.84      0.84      5000
weighted avg       0.84      0.84      0.84      5000


Confusion matrix:
[[2030  439]
 [ 362 2169]]


,text,true_label,pred_label,negative_prob,positive_prob,correct
0,"room finally organized it took 5 hours, 10 re...",positive,positive,1.136564e-06,9.999988e-01,True
1,last day today eeeep,negative,negative,9.999995e-01,4.592135e-07,True
2,@mikal_d that's what you get for being you,positive,positive,4.139937e-08,1.000000e+00,True
3,@1omarion cant respond too a normal girl,negative,negative,9.999843e-01,1.568955e-05,True
4,lol...watching Wedding Crashers...haven't watc...,positive,positive,6.402007e-05,9.999360e-01,True


## 14. Inspect mistakes

Use this to understand the errors the classifier is making.

In [17]:
mistakes_df = results_df[results_df["correct"] == False].copy()
print("Number of mistakes:", len(mistakes_df))
mistakes_df.head(20)

Number of mistakes: 801


,text,true_label,pred_label,negative_prob,positive_prob,correct
7,does notttt think the sushi in japan tastes ve...,negative,positive,2.335593e-09,1.000000e+00,False
12,@bryns awww poor you sir! hope you feel better...,positive,negative,9.999667e-01,3.321419e-05,False
14,Soot?? Do you Scoot? or Shoot? What a hoot.,positive,negative,9.972745e-01,2.725583e-03,False
15,@RonJeffries - so chet gets the speaker compen...,positive,negative,9.919379e-01,8.061991e-03,False
17,@livviclarke What is the school production thi...,positive,negative,9.999502e-01,4.985958e-05,False
18,@brigadeiro i knooow but i cannot resist AH8U...,negative,positive,4.198631e-05,9.999580e-01,False
19,blood diamonds are back http://tinyurl.com/nt...,negative,positive,4.400119e-05,9.999560e-01,False
21,"has just got into work, 9 hours to go then hom...",positive,negative,9.992903e-01,7.096704e-04,False
43,"@xomalese yeah, why doesn't your picture show ...",positive,negative,9.999976e-01,2.406103e-06,False
46,99 thieving last night,positive,negative,9.998928e-01,1.072088e-04,False


## 15. Save the classification-head LoRA model

This saves:

- LoRA adapter
- saved classifier head module
- tokenizer

The output directory is:

```python
./qwen3-0.6b-twitter-sentiment-classifier-lora_saved
```

In [18]:
clf_trainer.save_model(CLS_LORA_SAVED_DIR)
tokenizer.save_pretrained(CLS_LORA_SAVED_DIR)

print("Saved classifier LoRA model to:", CLS_LORA_SAVED_DIR)
print("Files:")
print(os.listdir(CLS_LORA_SAVED_DIR))

Saved classifier LoRA model to: ./qwen3-0.6b-twitter-sentiment-classifier-lora_saved
Files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin']


## 16. Single-text inference with the trained classifier

This function returns:

- predicted label
- probability for negative
- probability for positive

In [19]:
def get_model_device(model_obj):
    return next(model_obj.parameters()).device

@torch.no_grad()
def classify_tweet_with_head(model_obj, tokenizer_obj, tweet, max_length=MAX_LENGTH):
    model_obj.eval()
    device = get_model_device(model_obj)

    inputs = tokenizer_obj(
        tweet,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    ).to(device)

    outputs = model_obj(**inputs)
    logits = outputs.logits[0]
    probs = F.softmax(logits.float(), dim=-1)

    pred_id = int(torch.argmax(probs).item())
    pred_label = ID2LABEL[pred_id]

    return {
        "tweet": tweet,
        "prediction": pred_label,
        "probabilities": {
            ID2LABEL[i]: float(probs[i].item())
            for i in range(len(ID2LABEL))
        },
        "logits": {
            ID2LABEL[i]: float(logits.float()[i].item())
            for i in range(len(ID2LABEL))
        },
    }

custom_text = "I love this product. It works perfectly."
result = classify_tweet_with_head(clf_model, tokenizer, custom_text)
result

{'tweet': 'I love this product. It works perfectly.',
 'prediction': 'positive',
 'probabilities': {'negative': 1.7257828943684217e-08, 'positive': 1.0},
 'logits': {'negative': -9.0625, 'positive': 8.8125}}

## 17. Predict one row from validation or test set

Change:

```python
split_name
row_index
```

to inspect any example.

In [20]:
split_name = "validation"   # choose: "train", "validation", or "test"
row_index = 0

row = work_ds[split_name][row_index]

tweet = row["text"]
true_label = ID2LABEL[int(row["feeling"])]

result = classify_tweet_with_head(clf_model, tokenizer, tweet)

print("Tweet:", tweet)
print("True label:", true_label)
print("Prediction:", result["prediction"])
print("Probabilities:", result["probabilities"])
print("Correct:", result["prediction"] == true_label)

Tweet: @MylahMusic come on, one chicken finger won't hurt.  You know you wanna.
True label: positive
Prediction: positive
Probabilities: {'negative': 2.536018826049258e-07, 'positive': 0.9999997615814209}
Correct: True


## 18. Load the saved classification-head LoRA model from disk

Use this section in a new notebook/session when you only want inference.

Important:

For sequence-classification LoRA, we load:

1. the base `AutoModelForSequenceClassification`
2. the saved PEFT adapter using `PeftModel.from_pretrained`
3. the saved tokenizer

In [21]:
# Optional: free memory before loading from disk
try:
    del clf_trainer
except Exception:
    pass

try:
    del clf_model
except Exception:
    pass

try:
    del base_cls_model
except Exception:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleanup done.")

Memory cleanup done.


In [22]:
loaded_tokenizer = AutoTokenizer.from_pretrained(CLS_LORA_SAVED_DIR)

if loaded_tokenizer.pad_token is None:
    loaded_tokenizer.pad_token = loaded_tokenizer.eos_token

loaded_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=DTYPE,
)

loaded_base_model.config.pad_token_id = loaded_tokenizer.pad_token_id
loaded_base_model.config.problem_type = "single_label_classification"
loaded_base_model.config.use_cache = False

loaded_model = PeftModel.from_pretrained(
    loaded_base_model,
    CLS_LORA_SAVED_DIR,
)

loaded_model.to(DEVICE)
loaded_model.eval()

print("Loaded saved sequence-classification LoRA model.")
print("Device:", get_model_device(loaded_model))

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded saved sequence-classification LoRA model.
Device: cuda:0


## 19. Inference using the loaded model

This confirms that the saved model can be used without retraining.

In [23]:
custom_text = "The battery life is terrible and the product stopped working."
loaded_result = classify_tweet_with_head(
    loaded_model,
    loaded_tokenizer,
    custom_text,
)

loaded_result

{'tweet': 'The battery life is terrible and the product stopped working.',
 'prediction': 'negative',
 'probabilities': {'negative': 0.9999998807907104,
  'positive': 1.4908486889453343e-07},
 'logits': {'negative': 6.78125, 'positive': -8.9375}}

## 20. Evaluate loaded model on test set

This is useful when you reopen the notebook and only want to test a saved model.

In [24]:
@torch.no_grad()
def evaluate_loaded_classifier_on_raw_split(
    model_obj,
    tokenizer_obj,
    raw_split,
    sample_size=1000,
    batch_size=32,
    max_length=MAX_LENGTH,
):
    model_obj.eval()
    device = get_model_device(model_obj)

    if sample_size is not None:
        sample_size = min(sample_size, len(raw_split))
        raw_split = raw_split.shuffle(seed=SEED).select(range(sample_size))

    all_preds = []
    all_labels = []
    all_records = []

    for start in tqdm(range(0, len(raw_split), batch_size)):
        batch = raw_split[start:start + batch_size]

        texts = batch["text"]
        labels = batch["feeling"]

        inputs = tokenizer_obj(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)

        outputs = model_obj(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits.float(), dim=-1)

        preds = torch.argmax(probs, dim=-1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels)

        probs_np = probs.cpu().numpy()

        for text, true_id, pred_id, prob_row in zip(texts, labels, preds, probs_np):
            all_records.append({
                "text": text,
                "true_label": ID2LABEL[int(true_id)],
                "pred_label": ID2LABEL[int(pred_id)],
                "negative_prob": float(prob_row[0]),
                "positive_prob": float(prob_row[1]),
                "correct": int(true_id) == int(pred_id),
            })

    print("Accuracy:", accuracy_score(all_labels, all_preds))

    print("\nClassification report:")
    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=["negative", "positive"],
            zero_division=0,
        )
    )

    print("\nConfusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return pd.DataFrame(all_records)

loaded_eval_df = evaluate_loaded_classifier_on_raw_split(
    loaded_model,
    loaded_tokenizer,
    work_ds["test"],
    sample_size=1000,
    batch_size=32,
)

loaded_eval_df.head()

  0%|          | 0/32 [00:00<?, ?it/s]

Accuracy: 0.848

Classification report:
              precision    recall  f1-score   support

    negative       0.87      0.81      0.84       478
    positive       0.83      0.89      0.86       522

    accuracy                           0.85      1000
   macro avg       0.85      0.85      0.85      1000
weighted avg       0.85      0.85      0.85      1000


Confusion matrix:
[[386  92]
 [ 60 462]]


,text,true_label,pred_label,negative_prob,positive_prob,correct
0,Party time. http://twitpic.com/6qcs9,positive,positive,1.092371e-10,1.000000e+00,True
1,"@SimonJJennings ah, good point re: education m...",negative,positive,2.902312e-06,9.999971e-01,False
2,@adds9609 sure did!!!,positive,positive,6.615601e-08,9.999999e-01,True
3,@mcsnacktime Aww... poor Vince. I'll be your ...,negative,negative,9.999994e-01,5.896418e-07,True
4,"Depending on what time zone you are, good morn...",positive,positive,1.036413e-09,1.000000e+00,True


## 21. Recommended next experiments

Once this notebook runs successfully:

1. Increase `MAX_TRAIN_SAMPLES` or set it to `None`.
2. Try `MAX_LENGTH = 512` if tweets/context are longer.
3. Try batch size `48` or `64` if memory allows.
4. Train for `2–3` epochs and compare test accuracy.
5. Compare this sequence-classification-head model against your earlier generative SFT model.

For a pure classification use case, this classifier-head approach is cleaner, faster, and easier to evaluate than generative classification.